# 04 MIND Translation

This notebook explains how the public MIND dataset is translated into the same abstract ranking problem as the DSP demo.

Why MIND:
- it is clickstream-like and easy to explain,
- it has users, impression sets, and click labels,
- it maps naturally to the idea of "retrieve a pool, then rank items for a user".

The translation is intentionally lossy. It is not pretending that news articles are ads.
It is simply giving us a second dataset with real behavioral structure so we can compare evaluation quality outside the synthetic generator.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.common import read_jsonl
from data.huggingface_adapter import export_mind_translation
from experiments.evaluate import evaluate_mind_translation

OUTPUT_DIR = REPO_ROOT / 'data' / 'generated' / 'mind'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mapping = pd.DataFrame(
    [
        {'MIND field': 'user_id', 'Demo abstraction': 'user_id', 'How the demo uses it': 'request identity'},
        {'MIND field': 'history', 'Demo abstraction': 'user interests / segments', 'How the demo uses it': 'derive category counts and top segments'},
        {'MIND field': 'news category', 'Demo abstraction': 'campaign topic / interest dimension', 'How the demo uses it': 'score matching and segment mapping'},
        {'MIND field': 'impressions', 'Demo abstraction': 'candidate items', 'How the demo uses it': 'ranked slate with click labels'},
        {'MIND field': 'click label', 'Demo abstraction': 'relevance label', 'How the demo uses it': 'offline evaluation target'},
    ]
)
mapping

## Export A Translated Sample

The adapter downloads the raw MIND zip artifact from Hugging Face, parses `behaviors.tsv` and `news.tsv`, then emits:
- translated user rows,
- translated interaction rows,
- metadata describing the sampled split and size.

In [ ]:
translated_path = export_mind_translation(OUTPUT_DIR, split='train', sample_size=1500, variant='demo')
translated = pd.read_parquet(translated_path)
translated_users = read_jsonl(OUTPUT_DIR / 'mind_translated_users.jsonl')
metadata = pd.read_json(OUTPUT_DIR / 'mind_translation_metadata.json', typ='series')

metadata

## What The Translated Rows Look Like

The translated interaction table keeps the ranking problem intact: each user has an impression set, each impression has a label, and each item carries a coarse category.

In [ ]:
display(translated.head(10))
display(translated.groupby('label').size().rename('rows').reset_index())

## Example Translated User

The adapter turns recent news-category history into a lightweight interest vector and top segments.
That gives the MIND sample the same shape as the synthetic user profile used by the API.

In [ ]:
sample_user = translated_users[0]
pd.json_normalize(sample_user)

## Evaluation On The Translated Dataset

The MIND evaluation is expected to be weaker than the synthetic evaluation because the mapping is approximate and the scoring heuristic is much less aligned to the original data-generating process.
That is useful. It shows how the same retrieval-and-ranking stack behaves on a more realistic public dataset.

In [ ]:
results = evaluate_mind_translation(OUTPUT_DIR, top_k=5, sample_size=1500)
pd.Series(results)